In [0]:
%run ../utils

In [0]:
dbutils.widgets.text('date_val','')
date_val = dbutils.widgets.get('date_val')

In [0]:
date_val = calculate_date(date_val)

In [0]:
input_location = f'/mnt/silver-layer/{date_val}/earthquakedata'
df = spark.read.parquet(input_location)


In [0]:
print(df.columns)

In [0]:
location_url = "https://earthquake.usgs.gov/realtime/product/nearby-cities/ci41149896/ci/1746855057760/nearby-cities.json"

df = df.select("net", "code", "updated")

df = df.withColumn(
    "location_url",
    concat(
        lit("https://earthquake.usgs.gov/realtime/product/nearby-cities/"),
        col("net"),
        col("code"),
        lit("/"),
        col("net"),
        lit("/"),
        col("updated"),
        lit("/nearby-cities.json"),
    ),
).withColumn("row_id", concat(col("net"),col("code")))


In [0]:
location_url_id_lst = [{'location_url':x['location_url'],'row_id':x['row_id']} for x in df.select("location_url","row_id").collect()]

In [0]:
result_lst = []
cnt = 0
for data in location_url_id_lst:
    location_url = data['location_url']
    row_id = data['row_id']
    request_res = requests.get(location_url)
    if request_res.status_code == 200:
        if cnt <= 10:
            for val in request_res.json():
                val['row_id'] = row_id
                result_lst.append(val)
        else:
            break
    cnt += 1


In [0]:

# {'distance': 16, 'direction': 'W', 'name': 'Johannesburg, CA', 'longitude': -117.634722, 'latitude': 35.372778, 'population': None, 'row_id': 'ci41149840'}

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

schema = StructType([
    StructField('distance', IntegerType()),
    StructField('direction', StringType()),
    StructField('name', StringType()),
    StructField('longitude', FloatType()),
    StructField('latitude', FloatType()),
    StructField('population', IntegerType()),
    StructField('row_id', StringType())
])

df = spark.createDataFrame(result_lst, schema)

df.write.mode("overwrite").format('parquet').save(f'/mnt/bronze-layer/{date_val}/earthquake_nearby_cities')

In [0]:
df.display()